# Lab 2 — Ứng dụng Pre-trained Models với PyTorch

> **Practice 2 – Hands-on practice with pre-trained neural network architectures**

Notebook này thực hành toàn bộ quy trình **Transfer Learning** trên tập CIFAR-10 (60 000 ảnh, 10 lớp):

| Section | Nội dung |
|---------|----------|
| **0** | Setup & Config |
| **1** | Load & Visualize Data |
| **2** | Khám phá kiến trúc Pre-trained |
| **3** | Adapt model — thay lớp classifier |
| **4** | Train & Monitor với TensorBoard |
| **5** | So sánh models |
| **6** | Fine-tune — unfreeze backbone |
| **7** | Evaluate trên Test Set |

**Quy ước split dữ liệu:**
- **Train** (45 000) + **Validation** (5 000): dùng để huấn luyện và chọn checkpoint tốt nhất.
- **Test** (10 000): **chỉ đánh giá một lần duy nhất** ở Section 7 sau khi đã chốt cấu hình.


---
## Section 0 — Setup & Config


Đảm bảo đã cài: `pip install torch torchvision torchinfo tensorboard matplotlib pandas`

In [ ]:
from pathlib import Path
import sys

# Detect Colab and mount Drive so project code and saved artifacts persist.
try:
    import google.colab
    IS_COLAB = True
except ModuleNotFoundError:
    IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/Lab2')
    if not (ROOT / 'going_modular').is_dir():
        raise FileNotFoundError(
            f'Kh?ng t?m th?y {ROOT / "going_modular"}. '
            'H?y ??t th? m?c going_modular t?i MyDrive/Lab2.'
        )
else:
    ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Project root : {ROOT}')
print(f'Colab       : {IS_COLAB}')
print(f'Python path : {sys.path[0]}')


In [ ]:
import torch
from torch import nn
import torchvision.models as models
import matplotlib.pyplot as plt
import pandas as pd
from timeit import default_timer as timer

try:
    import torchinfo
except ImportError:
    raise ImportError('Cài torchinfo: pip install torchinfo')

try:
    from torch.utils.tensorboard import SummaryWriter
except ImportError:
    raise ImportError('Cài tensorboard: pip install tensorboard')

from going_modular import data_setup, engine, evaluate, model_builder, utils

print(f'PyTorch   : {torch.__version__}')
print(f'CUDA      : {torch.cuda.is_available()}')

In [ ]:
# ============================================================
# HYPERPARAMETERS ? ch?nh t?i ??y
# ============================================================
SEED        = 42
BATCH_SIZE  = 32
EPOCHS      = 5
LR          = 1e-3
VAL_SIZE    = 5_000
NUM_WORKERS = 0       # Windows: gi? 0

# Ch?y "train" m?t l?n ?? t?o artifacts; c?c l?n sau ch?n "reuse".
# reuse s? n?p checkpoint + history/config v? b? qua hu?n luy?n.
RUN_MODE = "train"  # "train" ho?c "reuse"
if RUN_MODE not in {"train", "reuse"}:
    raise ValueError("RUN_MODE ph?i l? 'train' ho?c 'reuse'.")

# Colab gi? dataset t?m trong runtime ?? tr?nh ??c ?nh tr?c ti?p t? Drive.
DATA_DIR = Path('/content/cifar10_data') if IS_COLAB else ROOT / 'data'
MODEL_DIR = ROOT / 'artifacts'
RUNS_DIR = ROOT / 'runs'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# -- Device & Seed --
utils.set_seed(SEED)
DEVICE = utils.get_device()

print(f'Device     : {DEVICE}')
print(f'Artifact dir: {MODEL_DIR}')
print(f'Seed={SEED} | Batch={BATCH_SIZE} | Epochs={EPOCHS} | LR={LR} | Mode={RUN_MODE}')


---
## Section 1 — Load & Visualize Data

### Bước 5: Prepare the Data

- Dùng preprocessing preset tương ứng với weights của từng kiến trúc: `VGG16_Weights.IMAGENET1K_V1` hoặc `ResNet18_Weights.IMAGENET1K_V1`.  
  Hai preset hiện tại có cùng cấu hình resize cạnh ngắn lên **256**, center crop **224×224**, chuyển tensor và ImageNet normalization, nên notebook có thể dùng chung DataLoader.
- Dùng `torch.utils.data.DataLoader` để tạo batches.

In [ ]:
train_dl, val_dl, test_dl, CLASS_NAMES = data_setup.create_train_val_test_dataloaders(
    batch_size   = BATCH_SIZE,
    data_dir     = str(DATA_DIR),
    transform_model = 'vgg16',  # VGG16/ResNet-18 weights hiện có cùng preprocessing
    val_size     = VAL_SIZE,
    seed         = SEED,
    num_workers  = NUM_WORKERS,
)

NUM_CLASSES = len(CLASS_NAMES)
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')
print(f'Train batches : {len(train_dl):>5}  (~{len(train_dl)*BATCH_SIZE:>6} samples)')
print(f'Val   batches : {len(val_dl):>5}  (~{len(val_dl)*BATCH_SIZE:>6} samples)')
print(f'Test  batches : {len(test_dl):>5}  (~{len(test_dl)*BATCH_SIZE:>6} samples)')

In [ ]:
# -- Kiểm tra shape & dtype của một batch --
images, labels = next(iter(train_dl))
print(f'Batch images : {images.shape}   # [B, C, H, W]')
print(f'Batch labels : {labels.shape}')
print(f'Dtype        : {images.dtype}')
print(f'Pixel range (ImageNet-normalized): [{images.min():.3f}, {images.max():.3f}]')

In [ ]:
# -- De-normalize để hiển thị ảnh --
# Tensor đã normalize theo ImageNet, cần đảo ngược trước khi plt.imshow()
from torchvision.models import VGG16_Weights
WEIGHT_TRANSFORM = VGG16_Weights.IMAGENET1K_V1.transforms()
IMAGENET_MEAN = torch.tensor(WEIGHT_TRANSFORM.mean).view(3, 1, 1)
IMAGENET_STD  = torch.tensor(WEIGHT_TRANSFORM.std).view(3, 1, 1)

def denormalize(t):
    """[C,H,W] normalized → [C,H,W] in [0,1]"""
    return (t * IMAGENET_STD + IMAGENET_MEAN).clamp(0.0, 1.0)

torch.manual_seed(SEED)
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, img, lbl in zip(axes.flat, images[:16], labels[:16]):
    ax.imshow(denormalize(img).permute(1, 2, 0))
    ax.set_title(CLASS_NAMES[lbl.item()], fontsize=9)
    ax.axis('off')
fig.suptitle('16 ảnh train (de-normalized, center crop 224×224)', fontsize=13)
plt.tight_layout()
plt.show()

---
## Section 2 ? Kh?m ph? Ki?n tr?c Pre-trained

`torchvision.models` cung c?p VGG16 v? ResNet-18 pretrained tr?n ImageNet. ? ch? ?? `train`, notebook t?i ImageNet weights. ? ch? ?? `reuse`, model ???c d?ng kh?ng t?i weights ImageNet v? checkpoint CIFAR-10 ?? l?u s? ???c n?p sau ??.


In [ ]:
utils.set_seed(SEED)

# Trong reuse, kh?i t?o ki?n tr?c kh?ng t?i weights ImageNet; checkpoint ??y ?? s? ???c n?p ? Section 4.
LOAD_PRETRAINED = RUN_MODE == "train"
model_v0 = model_builder.build_model(
    'v0', num_classes=NUM_CLASSES, freeze_backbone=True, pretrained=LOAD_PRETRAINED
)
model_v1 = model_builder.build_model(
    'v1', num_classes=NUM_CLASSES, freeze_backbone=True, pretrained=LOAD_PRETRAINED
)

# -- B?ng so s?nh params --
rows = []
for name, m in [('v0 ? VGG16', model_v0), ('v1 ? ResNet-18', model_v1)]:
    total = model_builder.count_total_params(m)
    trainable = model_builder.count_trainable_params(m)
    rows.append({'Model': name,
                 'Total params': f'{total:,}',
                 'Trainable': f'{trainable:,}',
                 'Frozen': f'{total - trainable:,}',
                 'Trainable %': f'{100*trainable/total:.1f}%'})

pd.DataFrame(rows).set_index('Model')


In [ ]:
# -- Xem kiến trúc V0 (VGG16) --
print('=' * 60)
print('V0 — VGG16 pretrained ImageNet  |  freeze_backbone=True')
print('=' * 60)
torchinfo.summary(
    model_v0,
    input_size=(2, 3, 224, 224),
    col_names=['input_size', 'output_size', 'num_params', 'trainable'],
    col_width=18,
    row_settings=['var_names'],
    verbose=0,
)

In [ ]:
# -- Xem kiến trúc V1 (ResNet-18) --
print('=' * 60)
print('V1 — ResNet-18 pretrained ImageNet  |  freeze_backbone=True')
print('=' * 60)
torchinfo.summary(
    model_v1,
    input_size=(2, 3, 224, 224),
    col_names=['input_size', 'output_size', 'num_params', 'trainable'],
    col_width=18,
    row_settings=['var_names'],
    verbose=0,
)

---
## Section 3 — Adapt Model cho CIFAR-10



**Chiến lược transfer learning:**

| Chiến lược | Mô tả | Khi nào dùng |
|---|---|---|
| **Freeze backbone** | Đóng băng toàn bộ pretrained layers, chỉ train classifier mới | Dataset nhỏ, tương đồng với ImageNet |
| **Fine-tune** | Unfreeze một số layer cuối + train cả classifier | Dataset lớn hơn, muốn adapt thêm |
| **Full training** | Train toàn bộ mạng | Dataset rất lớn, rất khác ImageNet |

**Lý do freeze BatchNorm trong ResNet:**  
BatchNorm layers lưu `running_mean` và `running_var` từ ImageNet. Nếu để `model.train()` bình thường, các thống kê này sẽ bị ghi đè bởi CIFAR-10 batches — làm mất thông tin pretrained. Giải pháp: giữ backbone ở `eval()` mode.

In [ ]:
# -- Verify output shape: phải là [B, 10] cho CIFAR-10 --
dummy = torch.randn(2, 3, 224, 224)
with torch.inference_mode():
    out_v0 = model_v0(dummy)
    out_v1 = model_v1(dummy)

assert out_v0.shape == (2, NUM_CLASSES), f'v0 shape sai: {out_v0.shape}'
assert out_v1.shape == (2, NUM_CLASSES), f'v1 shape sai: {out_v1.shape}'
print(f'v0 (VGG16)    output: {out_v0.shape}  ✓')
print(f'v1 (ResNet18) output: {out_v1.shape}  ✓')

In [ ]:
# -- Xem lớp classifier đã được thay --
print('V0 classifier (lớp cuối VGG16):')
print(model_v0.model.classifier[-1])   # Linear(4096 -> 10)
print()
print('V1 classifier (fc layer ResNet-18):')
print(model_v1.model.fc)          # Linear(512 -> 10)

---
## Section 4 ? Train & Monitor v?i TensorBoard

- **Train mode:** hu?n luy?n VGG16 v? ResNet-18, ??nh gi? validation m?i epoch, l?u checkpoint t?t nh?t c?ng history/config v?o `artifacts/`.
- **Reuse mode:** n?p l?i checkpoint v? history ?? l?u, b? qua ph?n hu?n luy?n ?? ti?p t?c Section 5?7.
- **Loss:** `CrossEntropyLoss`; **optimizer:** Adam ch? nh?n c?c tham s? c? `requires_grad=True`.
- **TensorBoard:** ghi loss/accuracy train v? validation trong m?i epoch.

??t `RUN_MODE = "train"` cho l?n ch?y ??u. Sau khi c?c artifacts ?? l?u ??y ??, ??i th?nh `RUN_MODE = "reuse"` ? Section 0 v? ch?y notebook l?i t? ??u. Tr?n Colab, artifacts ???c l?u tr?n Google Drive; kh?ng b? m?t khi runtime reset.

Sau khi train, m? terminal v? ch?y:
```
tensorboard --logdir runs/
```
Truy c?p: http://localhost:6006


In [ ]:
def log_run_config(writer, config: dict):
    """Ghi run config vào TensorBoard text tab."""
    lines = [f'`{k}` = `{v}`' for k, v in config.items()]
    writer.add_text('run_config', '  \n'.join(lines), global_step=0)

loss_fn = nn.CrossEntropyLoss()

### 4.1 Train V0 — VGG16 (frozen backbone)

In [ ]:
ARTIFACT_V0 = MODEL_DIR / 'v0_vgg16'
CHECKPOINT_V0 = str(ARTIFACT_V0 / 'best_model.pth')
CONFIG_V0 = {
    'model': 'VGG16', 'weights': 'ImageNet1K_V1', 'num_classes': NUM_CLASSES,
    'freeze_backbone': True, 'batch_size': BATCH_SIZE,
    'lr': LR, 'epochs': EPOCHS, 'seed': SEED,
}
print(f"VGG16 artifacts: {ARTIFACT_V0}")


In [ ]:
print('-' * 55)
print(f'VGG16 | mode={RUN_MODE} | freeze_backbone=True')
print('-' * 55)

if RUN_MODE == 'train':
    utils.set_seed(SEED)
    optimizer_v0 = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model_v0.parameters()), lr=LR
    )
    writer_v0 = SummaryWriter(log_dir=str(RUNS_DIR / 'v0_vgg16'))
    log_run_config(writer_v0, CONFIG_V0)

    start_v0 = timer()
    try:
        history_v0 = engine.train(
            model=model_v0,
            train_dataloader=train_dl,
            val_dataloader=val_dl,
            optimizer=optimizer_v0,
            loss_fn=loss_fn,
            epochs=EPOCHS,
            device=DEVICE,
            writer=writer_v0,
            checkpoint_path=CHECKPOINT_V0,
            frozen_backbone=None,
        )
    finally:
        writer_v0.close()
    time_v0 = timer() - start_v0
    CONFIG_V0['train_seconds'] = time_v0
    utils.save_training_metadata(history_v0, str(ARTIFACT_V0), CONFIG_V0)
else:
    model_v0, history_v0, saved_config_v0 = utils.load_training_artifacts(
        model_v0, str(ARTIFACT_V0), device=DEVICE
    )
    time_v0 = float(saved_config_v0.get('train_seconds', 0.0))
    print(f"Loaded saved config: {saved_config_v0}")

best_v0 = max(history_v0['val_acc'])
print(f'VGG16 ready | train time: {time_v0:.1f}s | best val_acc: {best_v0:.4f}')


### 4.2 Train V1 — ResNet-18 (frozen backbone)

**Lưu ý BatchNorm**: truyền `frozen_backbone=model_v1.model` để engine giữ backbone ở `eval()` mode mỗi bước train.

In [ ]:
ARTIFACT_V1 = MODEL_DIR / 'v1_resnet18'
CHECKPOINT_V1 = str(ARTIFACT_V1 / 'best_model.pth')
CONFIG_V1 = {
    'model': 'ResNet-18', 'weights': 'ImageNet1K_V1', 'num_classes': NUM_CLASSES,
    'freeze_backbone': True, 'batch_size': BATCH_SIZE,
    'lr': LR, 'epochs': EPOCHS, 'seed': SEED,
}
print(f"ResNet-18 artifacts: {ARTIFACT_V1}")


In [ ]:
print('-' * 55)
print(f'ResNet-18 | mode={RUN_MODE} | freeze_backbone=True')
print('-' * 55)

if RUN_MODE == 'train':
    utils.set_seed(SEED)
    optimizer_v1 = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model_v1.parameters()), lr=LR
    )
    writer_v1 = SummaryWriter(log_dir=str(RUNS_DIR / 'v1_resnet18'))
    log_run_config(writer_v1, CONFIG_V1)

    start_v1 = timer()
    try:
        history_v1 = engine.train(
            model=model_v1,
            train_dataloader=train_dl,
            val_dataloader=val_dl,
            optimizer=optimizer_v1,
            loss_fn=loss_fn,
            epochs=EPOCHS,
            device=DEVICE,
            writer=writer_v1,
            checkpoint_path=CHECKPOINT_V1,
            frozen_backbone=model_v1.model,
        )
    finally:
        writer_v1.close()
    time_v1 = timer() - start_v1
    CONFIG_V1['train_seconds'] = time_v1
    utils.save_training_metadata(history_v1, str(ARTIFACT_V1), CONFIG_V1)
else:
    model_v1, history_v1, saved_config_v1 = utils.load_training_artifacts(
        model_v1, str(ARTIFACT_V1), device=DEVICE
    )
    time_v1 = float(saved_config_v1.get('train_seconds', 0.0))
    print(f"Loaded saved config: {saved_config_v1}")

best_v1 = max(history_v1['val_acc'])
print(f'ResNet-18 ready | train time: {time_v1:.1f}s | best val_acc: {best_v1:.4f}')


---
## Section 5 — So sánh V0 vs V1



Trực quan hóa loss/accuracy curves để so sánh tốc độ học và overfitting.

In [ ]:
# -- Bảng tóm tắt kết quả training --
summary_data = {
    'Model'           : ['v0 — VGG16', 'v1 — ResNet-18'],
    'Best val_acc'    : [f'{best_v0:.4f}', f'{best_v1:.4f}'],
    'Train time (s)'  : [f'{time_v0:.1f}', f'{time_v1:.1f}'],
    'Trainable params': [
        f"{model_builder.count_trainable_params(model_v0):,}",
        f"{model_builder.count_trainable_params(model_v1):,}",
    ],
}
pd.DataFrame(summary_data).set_index('Model')

In [ ]:
# -- Loss & Accuracy curves --
evaluate.plot_history(
    histories={
        'v0 — VGG16'    : history_v0,
        'v1 — ResNet-18': history_v1,
    },
    show=True,
)

---
## Section 6 — Fine-tune Experiment



**Chiến lược fine-tune:**
1. Bắt đầu từ checkpoint đã train (Section 4).
2. **Unfreeze** một số layer cuối của backbone.
3. Train thêm với **LR nhỏ hơn** (tránh phá vỡ pretrained weights).

Ở đây ta fine-tune **ResNet-18** — unfreeze `layer4` + `fc`.

In [ ]:
# Kh?i t?o ki?n tr?c r?i n?p checkpoint t?t nh?t c?a ResNet-18.
model_v1_ft = model_builder.build_model(
    'v1', num_classes=NUM_CLASSES, freeze_backbone=True, pretrained=False
)
model_v1_ft = utils.load_model(model_v1_ft, CHECKPOINT_V1, device=DEVICE)

# -- Unfreeze layer4 v? fc c?a ResNet-18 --
for name, param in model_v1_ft.model.named_parameters():
    if 'layer4' in name or 'fc' in name:
        param.requires_grad = True

total_ft = model_builder.count_total_params(model_v1_ft)
trainable_ft = model_builder.count_trainable_params(model_v1_ft)
print(f'Sau unfreeze layer4+fc:')
print(f'  Total     : {total_ft:,}')
print(f'  Trainable : {trainable_ft:,}  ({100*trainable_ft/total_ft:.1f}%)')
print(f'  Frozen    : {total_ft - trainable_ft:,}')


In [ ]:
LR_FINETUNE = 1e-4
EPOCHS_FT = 3
ARTIFACT_FT = MODEL_DIR / 'v1_resnet18_finetune'
CHECKPOINT_FT = str(ARTIFACT_FT / 'best_model.pth')
CONFIG_FT = {
    'model': 'ResNet-18 fine-tuned', 'weights': 'ImageNet1K_V1',
    'starting_checkpoint': 'v1_resnet18/best_model.pth',
    'unfrozen_layers': ['layer4', 'fc'], 'num_classes': NUM_CLASSES,
    'batch_size': BATCH_SIZE, 'lr': LR_FINETUNE,
    'epochs': EPOCHS_FT, 'seed': SEED,
}

if RUN_MODE == 'train':
    utils.set_seed(SEED)
    optimizer_ft = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model_v1_ft.parameters()),
        lr=LR_FINETUNE,
    )
    writer_ft = SummaryWriter(log_dir=str(RUNS_DIR / 'v1_resnet18_finetune'))
    log_run_config(writer_ft, CONFIG_FT)

    print('-' * 55)
    print(f'Fine-tuning ResNet-18 layer4+fc | LR={LR_FINETUNE}')
    print('-' * 55)

    start_ft = timer()
    try:
        history_ft = engine.train(
            model=model_v1_ft,
            train_dataloader=train_dl,
            val_dataloader=val_dl,
            optimizer=optimizer_ft,
            loss_fn=loss_fn,
            epochs=EPOCHS_FT,
            device=DEVICE,
            writer=writer_ft,
            checkpoint_path=CHECKPOINT_FT,
            frozen_backbone=None,
        )
    finally:
        writer_ft.close()
    time_ft = timer() - start_ft
    CONFIG_FT['train_seconds'] = time_ft
    utils.save_training_metadata(history_ft, str(ARTIFACT_FT), CONFIG_FT)
else:
    model_v1_ft, history_ft, saved_config_ft = utils.load_training_artifacts(
        model_v1_ft, str(ARTIFACT_FT), device=DEVICE
    )
    time_ft = float(saved_config_ft.get('train_seconds', 0.0))
    print(f"Loaded saved fine-tuning config: {saved_config_ft}")

best_ft = max(history_ft['val_acc'])
print(f'Fine-tuned model ready | train time: {time_ft:.1f}s | best val_acc: {best_ft:.4f}')


In [ ]:
# -- So sánh: frozen vs fine-tuned --
print('\nSo sánh ResNet-18:')
print(f'  Frozen backbone  : val_acc = {best_v1:.4f}')
print(f'  Fine-tuned layer4: val_acc = {best_ft:.4f}')
print(f'  Cải thiện        : {(best_ft - best_v1)*100:+.2f}%')

evaluate.plot_history(
    histories={
        'v1 frozen'   : history_v1,
        'v1 fine-tune': history_ft,
    },
    show=True,
)

---
## Section 7 — Evaluate trên Test Set



> ⚠️ **Chỉ chạy section này SAU KHI đã chốt cấu hình model.**  
> Test set phải là tập dữ liệu "chưa từng thấy" — không được dùng để quyết định hyperparameters.

In [ ]:
# -- Đánh giá cả hai model trên test set --
result_v0 = engine.eval_model(model_v0, test_dl, loss_fn, device=DEVICE)
result_v1 = engine.eval_model(model_v1, test_dl, loss_fn, device=DEVICE)
result_ft = engine.eval_model(model_v1_ft, test_dl, loss_fn, device=DEVICE)

test_results = pd.DataFrame([
    {'Run': 'v0 — VGG16',          **result_v0},
    {'Run': 'v1 — ResNet-18',       **result_v1},
    {'Run': 'v1 Fine-tuned layer4', **result_ft},
]).set_index('Run')

print('=== TEST SET RESULTS ===')
print(test_results.to_string())

In [ ]:
# -- Confusion matrix cho model tốt nhất --
# (Dùng v1 fine-tuned)
y_pred, y_true = evaluate.predict_dataloader(model_v1_ft, test_dl, device=DEVICE)
cm = evaluate.compute_confusion_matrix(y_pred, y_true, num_classes=NUM_CLASSES)

evaluate.plot_confusion_matrix(
    cm,
    class_names=CLASS_NAMES,
    title='Confusion Matrix — V1 ResNet-18 Fine-tuned (Test Set)',
    show=True,
)

In [ ]:
# -- Per-class accuracy --
class_acc = evaluate.per_class_accuracy(cm)

df_class = pd.DataFrame({
    'Class'   : CLASS_NAMES,
    'Accuracy': [f'{a:.3f}' for a in class_acc.tolist()],
}).sort_values('Accuracy')

print('Per-class accuracy (thấp → cao):')
print(df_class.to_string(index=False))

In [ ]:
# -- Top 5 nhầm lẫn phổ biến nhất --
top5 = evaluate.top_confusions(cm, CLASS_NAMES, k=5)
print('Top 5 nhầm lẫn:')
print(f'  {"True":>12} → {"Predicted":<12}  Count')
print('  ' + '-' * 38)
for true_cls, pred_cls, count in top5:
    print(f'  {true_cls:>12} → {pred_cls:<12}  {count}')

In [ ]:
# -- Visualize một số dự đoán sai --
wrong_mask = (y_pred != y_true)
wrong_indices = wrong_mask.nonzero(as_tuple=True)[0][:9]

# Lấy ảnh gốc từ test_dl (cần rebuild để có ảnh de-normalized)
all_imgs, all_true = [], []
for imgs, lbls in test_dl:
    all_imgs.append(imgs)
    all_true.append(lbls)
all_imgs = torch.cat(all_imgs)

sample_imgs  = [denormalize(all_imgs[i]) for i in wrong_indices]
sample_true  = [y_true[i].item() for i in wrong_indices]
sample_pred  = [y_pred[i].item() for i in wrong_indices]

evaluate.plot_sample_predictions(
    images      = sample_imgs,
    true_labels = sample_true,
    pred_labels = sample_pred,
    class_names = CLASS_NAMES,
    ncols       = 3,
    show        = True,
)

---
## Kết luận

### Những gì đã thực hành:

1. **Load pre-trained model** từ `torchvision.models` với `weights=...IMAGENET1K_V1`
2. **Explore architecture** — dùng `torchinfo.summary()` để xem layers và params
3. **Adapt model** — thay classifier cuối để phân loại 10 lớp CIFAR-10
4. **Freeze backbone** — chỉ train classifier mới; giải quyết vấn đề BatchNorm
5. **Train & Monitor** — dùng TensorBoard để theo dõi loss/accuracy theo epoch
6. **Fine-tune** — unfreeze layer4, train thêm với LR nhỏ hơn
7. **Evaluate** — confusion matrix, per-class accuracy, visualize predictions


